# Notebook 02 — Fine-tune Student Model
**Member: B** | Runtime > Change runtime type > **T4 GPU** before running.

| Experiment | `data.mode` | `training.output_dir` |
|---|---|---|
| Exp 2 (adaptation)   | `adaptation`   | `results/checkpoints/adaptation` |
| Exp 3 (distillation) | `distillation` | `results/checkpoints/distillation` |
| Exp 4 (filtered)     | `filtered`     | `results/checkpoints/filtered` |

Ablations — change one field in `configs/finetune.yaml` per run:
- A2 rank sweep: `lora.rank` ∈ {4, 8, 16, 32}
- A3 data size: `data.train_fraction` ∈ {0.1, 0.25, 0.5, 1.0}

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
token = 'ghp_xxxxxxxxxxxx'  # paste your GitHub token each session
if not os.path.exists('/content/SC4001-Group133'):
    os.system(f'git clone https://{token}@github.com/TIFFMUN/SC4001-Group133.git /content/SC4001-Group133')
%cd /content/SC4001-Group133
!pip install -r requirements.txt -q

In [ ]:
# Redirect HF cache + data/ + results/ to Drive — run before any HF imports
import sys
sys.path.insert(0, '/content/SC4001-Group133')
from src.setup_cache import setup
setup()

In [ ]:
import torch
assert torch.cuda.is_available(), 'No GPU — enable via Runtime > Change runtime type > T4 GPU'
print(f'GPU: {torch.cuda.get_device_name(0)} | VRAM: {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB')

In [ ]:
# Check config before training — edit configs/finetune.yaml to switch experiments
import yaml
with open('configs/finetune.yaml') as f:
    cfg = yaml.safe_load(f)
print(f"Mode: {cfg['data']['mode']} | Rank: {cfg['lora']['rank']} | Fraction: {cfg['data']['train_fraction']}")
print(f"Output: {cfg['training']['output_dir']} | Epochs: {cfg['training']['num_epochs']}")

In [ ]:
# Checkpoints save every 200 steps to results/ (symlinked to Drive)
# If session dies: re-run cells above, then re-run this — resumes automatically
from src.training.finetune import train
train()

In [ ]:
# Confirm checkpoint on Drive
drive_path = f"/content/drive/MyDrive/SC4001/{cfg['training']['output_dir']}"
print(os.listdir(drive_path) if os.path.exists(drive_path) else f'Not found: {drive_path}')